In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
# 1. 准备数据
items = [
    {
        "id": "用户A",
        "content": "用户A的手机号是13800001234"
    },
    {
        "id": "用户B",
        "content": "用户B的手机号是13800005678"
    },
    {
        "id": "订单1001",
        "content": "订单1001编号203011010001"
    },
    {
        "id": "订单1002",
        "content": "订单1002编号203011010123"
    }
]

In [ ]:
# 2. 加载 Embedding 模型
model = SentenceTransformer(
    "BAAI/bge-small-zh-v1.5"
)

texts = [
    item["content"] for item in items
]


embeddings = model.encode(
    texts,
    normalize_embeddings=True
)

# 转换成 FAISS 要求的 float32
embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

print("向量维度：", dimension)

In [ ]:
# 3. 创建 FAISS 索引

# 归一化后的向量使用内积
# 等价于 Cosine Similarity
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print(f"已添加 {len(items)} 条向量")

In [ ]:
# 4. 查询
query_text = "用户A的电话号码是多少?"

query_vector = model.encode(
    [query_text],
    normalize_embeddings=True
)

query_vector = np.asarray(
    query_vector,
    dtype="float32"
)

In [ ]:
# 5. Top-K 相似度搜索
k = 3

scores, indices = index.search(
    query_vector,
    k
)

In [ ]:
# 6. 输出结果
print(f"\n查询:{query_text}")
print(f"Top-{k} 相似结果：")

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):

    item = items[int(idx)]

    print(
        f"{rank}. "
        f"score={score:.4f} | "
        f"id={item['id']} | "
        f"content={item['content']}"
    )